# Xeno Data Analyst Assignment — Comm-Log Reconciliation

Finance says the target_base for merchant 501 in October 2026 is 22, I’ll start with the raw data and try to figure out how we get to that number.

In [1]:
import sqlite3
import pandas as pd

In [2]:
conn = sqlite3.connect("comm_log.db")

## Getting the tables

In [8]:
pd.read_sql_query(
    """
    select name from sqlite_master where type='table';
    """, conn
)

,name
0,campaign
1,communication_log


In [9]:
pd.read_sql_query(
    """
    select * from campaign; """, conn
)

,id,merchant_id,parent_id,name,creation_status,processing_status
0,9001,501,NaN,Diwali Cart Recovery - Wave 1,approved,processed
1,9002,501,9001.0,Diwali Cart Recovery - Retry A,approved,processed
2,9003,501,9002.0,Diwali Cart Recovery - Retry B,approved,processed
3,9004,501,9001.0,Diwali Cart Recovery - Retry C (pending),approval_awaiting,processed
4,9101,501,NaN,Diwali Flash Sale - Standalone,approved,processed
5,9201,501,NaN,Diwali Wave 2,approved,processed
6,9202,501,9201.0,Diwali Wave 2 - Retry,approved,processed


In [10]:
pd.read_sql_query(
    """
    select * from communication_log;""", conn
)

,id,merchant_id,communication_id,customer_id,communication_type,delivery_status,sent_time,scheduled_time,credit_used,channel
0,1,501,9001,C1,2,900,2026-10-03 10:00:00,2026-10-03 10:00:00,1,sms
1,2,501,9001,C2,2,1100,2026-10-03 10:00:00,2026-10-03 10:00:00,1,sms
2,3,501,9002,C2,2,900,2026-10-04 10:00:00,2026-10-04 10:00:00,1,sms
3,4,501,9001,C3,2,1100,2026-10-03 10:00:00,2026-10-03 10:00:00,1,sms
4,5,501,9002,C3,2,1100,2026-10-04 10:00:00,2026-10-04 10:00:00,1,sms
5,6,501,9003,C3,2,900,2026-10-05 10:00:00,2026-10-05 10:00:00,1,sms
6,7,501,9001,C4,2,900,2026-10-03 10:00:00,2026-10-03 10:00:00,1,sms
7,8,501,9001,C5,2,900,2026-10-03 10:00:00,2026-10-03 10:00:00,1,sms
8,9,501,9001,C6,2,900,2026-10-03 10:00:00,2026-10-03 10:00:00,1,sms
9,10,501,9001,C7,2,900,2026-10-03 10:00:00,2026-10-03 10:00:00,1,sms


## number of all send records

In [11]:
pd.read_sql_query(
    """
    select count(*) as total_count
    from communication_log; """,conn)

,total_count
0,30


Total count came to be 30 but the finance's reported 22. I'll get the unique customers out of these.

In [12]:
pd.read_sql_query(
    """
    select count(distinct customer_id) as unique_customers 
    from communication_log;
    """,
    conn
)

,unique_customers
0,25


This shows that there total 25 unique customers, I'll look into the repeated customers to know the actual difference

In [13]:
pd.read_sql_query(
    """
    select customer_id,count(*) as send_count
    from communication_log
    group by customer_id
    having count(*) > 1
    order by send_count desc, customer_id;
    """,
    conn
)

,customer_id,send_count
0,C3,3
1,C2,2
2,C20,2
3,D1,2


I will dig deeper to find why there were repeated send commands for these customers

In [14]:
pd.read_sql_query(
    """
    select
        customer_id,
        communication_id,
        delivery_status,
        sent_time
    from communication_log
    where customer_id in ('C2', 'C3', 'C20', 'D1')
    order by customer_id, sent_time;
    """,
    conn
)

,customer_id,communication_id,delivery_status,sent_time
0,C2,9001,1100,2026-10-03 10:00:00
1,C2,9002,900,2026-10-04 10:00:00
2,C20,9101,900,2026-10-10 10:00:00
3,C20,9101,900,2026-10-20 10:00:00
4,C3,9001,1100,2026-10-03 10:00:00
5,C3,9002,1100,2026-10-04 10:00:00
6,C3,9003,900,2026-10-05 10:00:00
7,D1,9201,1100,2026-10-07 10:00:00
8,D1,9202,900,2026-10-08 10:00:00


from t